# TechLead-style SQL для дашборда (Impala memory-safe)

Тетрадка содержит:
1. Проверку количества уникальных ИНН в Excel за 3 месяца.
2. Проверку количества уникальных ИНН в витрине техлида без фильтра `agr_id is not null`.
3. Чистый SQL-скрипт (CTE-стиль) для формирования данных дашборда с учетом ограничений памяти Impala.

In [ ]:
import re
from decimal import Decimal, InvalidOperation
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None

In [ ]:
# Параметры
base_excel_dir = '/home/jovyan/documents/Equaring/Data'
month_files = {
    '2026-01': '01_Январь_2026.xlsx',
    '2026-02': '02_Февраль_2026.xlsx',
    '2026-03': '03_Марта_2026.xlsx',
}

excel_inn_col = 'ИНН'
techlead_mart_table = 'sandbox_ai.stg__chesnov_aef__sa_acquiring_merchants'

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
with imp:
    imp.execute(f'invalidate metadata {techlead_mart_table}')
print('Impala connected, metadata invalidated')

## 1) Уникальные ИНН в Excel и в витрине техлида (без `agr_id is not null`)

In [ ]:
excel_stats = []
excel_sets = {}

for month_key, file_name in month_files.items():
    path = f"{base_excel_dir}/{file_name}"
    raw = pd.read_excel(path, header=None)

    header_row = 0
    for i in range(min(30, len(raw))):
        vals = set([str(x).strip() for x in raw.iloc[i].tolist()])
        if ('ИНН' in vals) and (('ID договора' in vals) or ('Номер договора' in vals)):
            header_row = i
            break

    df = pd.read_excel(path, header=header_row)
    df.columns = [str(c).strip() for c in df.columns]
    if excel_inn_col not in df.columns:
        raise ValueError(f'[{month_key}] В Excel не найдена колонка ИНН')

    s = (
        df[excel_inn_col]
        .astype(str)
        .str.strip()
        .str.replace(' ', '', regex=False)
        .str.replace('\xa0', '', regex=False)
    )
    s = s.where(~s.str.lower().isin(['nan', 'none', 'null', '']))
    s = s.apply(lambda x: re.sub(r'\\.0+$', '', format(Decimal(x.replace(',', '.')), 'f')) if isinstance(x, str) and re.search(r'[eE][+-]?\\d+$', x.replace(',', '.')) else x)
    s = s.where(s.str.match(r'^\\d+$', na=False))
    inn_set = set(s.dropna().tolist())

    excel_sets[month_key] = inn_set
    excel_stats.append({'month': month_key, 'source': 'excel', 'header_row': header_row, 'unique_inn': len(inn_set)})

mart_stats = []
mart_sets = {}

for month_key in month_files.keys():
    month_start = pd.to_datetime(month_key + '-01').strftime('%Y-%m-%d')

    with imp:
        mart_df = imp.fetch(f"""
            select distinct cast(inn as string) as inn
            from {techlead_mart_table}
            where cast(d_valid_from as date) <= cast('{month_start}' as date)
              and (d_valid_to is null or cast(d_valid_to as date) >= cast('{month_start}' as date))
              and inn is not null
        """)

    inn_set = set(mart_df['inn'].dropna().astype(str).str.strip().tolist())
    mart_sets[month_key] = inn_set
    mart_stats.append({'month': month_key, 'source': 'techlead_mart_wo_agrid_not_null', 'unique_inn': len(inn_set)})

display(pd.DataFrame(excel_stats + mart_stats).sort_values(['month', 'source']))

In [ ]:
# Несостыковки множеств INN по месяцам (Excel vs витрина техлида)
diff_rows = []
for month_key in month_files.keys():
    ex = excel_sets[month_key]
    dm = mart_sets[month_key]
    diff_rows.append({
        'month': month_key,
        'common_inn': len(ex & dm),
        'only_excel_inn': len(ex - dm),
        'only_mart_inn': len(dm - ex),
    })

display(pd.DataFrame(diff_rows))

## 2) SQL-скрипт как у техлида (без Python-функций, с учетом памяти Impala)

Идея memory-safe:
- сначала строим узкий SA-периметр (`sa_base`) на 1-е число месяца;
- транзакции фильтруем по DAG-условиям до агрегации;
- убираем тяжелый `count(distinct ...)` на широком join, делаем dedup в отдельном CTE;
- считаем метрики на уровне INN, потом итоговую сводку.

In [ ]:
# Параметры месяца для SQL
month_start = '2026-02-01'
month_end = (pd.to_datetime(month_start) + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')

sql_script = f"""
with params as (
    select cast('{month_start}' as date) as month_start,
           cast('{month_end}' as date) as month_end
),

sa_base as (
    select
        cast(c.c_inn as string) as inn,
        cast(a.n_cmp_client as string) as n_cmp_client,
        cast(a.n_agr as string) as n_agr
    from ods_alpha.scd1_agreements a
    join ods_alpha.scd1_companies c
      on c.n_cmp = a.n_cmp_client
    join params p on 1 = 1
    where a.acq_class = 'SA'
      and cast(a.d_valid_from as date) <= p.month_start
      and (a.d_valid_to is null or cast(a.d_valid_to as date) >= p.month_start)
      and c.c_inn is not null
),

inn_agr as (
    select distinct inn, n_agr
    from sa_base
),

inn_cmp as (
    select distinct inn, n_cmp_client
    from sa_base
),

trx_filtered as (
    select
        t.n_trx,
        to_date(cast(t.d_trx_orig as timestamp)) as trx_dt
    from ods_alpha.scd1_trx t
    join params p on 1 = 1
    where to_date(cast(t.d_trx_orig as timestamp)) between p.month_start and p.month_end
      and t.c_trx_class = 'SA'
      and t.c_trx_type = 'S01'
      and t.c_nter is not null
      and t.ods_deleted_flg <> '1'
      and t.cf_trx_stat <> 'R'
),

trx_agr_dedup as (
    select
        cast(ta.n_agr as string) as n_agr,
        ta.n_trx
    from ods_alpha.scd1_trx_acq ta
    join trx_filtered tf
      on tf.n_trx = ta.n_trx
    group by cast(ta.n_agr as string), ta.n_trx
),

trx_by_inn as (
    select
        ia.inn,
        count(*) as trx_cnt
    from inn_agr ia
    left join trx_agr_dedup t
      on t.n_agr = ia.n_agr
    group by ia.inn
),

retl_by_cmp as (
    select
        cast(m.n_cmp as string) as n_cmp_client,
        count(distinct cast(m.c_nmrc as string)) as retl_cnt
    from ods_alpha.scd1_merchants m
    where m.c_nmrc is not null
    group by cast(m.n_cmp as string)
),

retl_by_inn as (
    select
        ic.inn,
        sum(coalesce(r.retl_cnt, 0)) as retl_cnt
    from inn_cmp ic
    left join retl_by_cmp r
      on r.n_cmp_client = ic.n_cmp_client
    group by ic.inn
),

term_by_cmp as (
    select
        cast(m.n_cmp as string) as n_cmp_client,
        count(distinct cast(t.c_nter as string)) as term_cnt
    from ods_alpha.scd1_merchants m
    join ods_alpha.scd1_pos_terminals t
      on t.c_nmrc = m.c_nmrc
    join params p on 1 = 1
    where t.c_nter is not null
      and cast(t.d_ter_install as date) <= p.month_end
      and (t.d_ter_close is null or cast(t.d_ter_close as date) >= p.month_start)
    group by cast(m.n_cmp as string)
),

term_by_inn as (
    select
        ic.inn,
        sum(coalesce(t.term_cnt, 0)) as term_cnt
    from inn_cmp ic
    left join term_by_cmp t
      on t.n_cmp_client = ic.n_cmp_client
    group by ic.inn
),

metrics_per_inn as (
    select
        i.inn,
        coalesce(trx.trx_cnt, 0) as trx_cnt,
        coalesce(retl.retl_cnt, 0) as retl_cnt,
        coalesce(term.term_cnt, 0) as term_cnt
    from (select distinct inn from sa_base) i
    left join trx_by_inn trx on trx.inn = i.inn
    left join retl_by_inn retl on retl.inn = i.inn
    left join term_by_inn term on term.inn = i.inn
),

summary_month as (
    select
        cast('{month_start}' as date) as month_start,
        count(*) as unique_clients,
        sum(trx_cnt) as trx_cnt_total,
        sum(retl_cnt) as retl_cnt_total,
        sum(term_cnt) as term_cnt_total
    from metrics_per_inn
)

-- 1) Клиентский слой
select *
from metrics_per_inn
order by inn;

-- 2) Месячная сводка
select *
from summary_month;
"""

print(sql_script)

## 3) Выполнение SQL частями (без перегруза памяти)

Рекомендуется:
1. Сначала выполнить только клиентский слой `metrics_per_inn`.
2. Затем отдельным запросом `summary_month`.
3. Если пул перегружен: запускать по одному месяцу и вне пиковых часов.